# Dynamic Batching

## Learning Objectives
1. Understand static vs dynamic vs continuous batching
2. Implement a dynamic batching scheduler
3. Analyze throughput vs latency trade-offs
4. Compare batching strategies under load

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

np.random.seed(42)
print('Dynamic batching scheduler simulation')

## Level 1: Basic Static Batching

In [ ]:
# Static batching: wait until batch_size requests
def simulate_static_batch(request_arrivals, batch_size=32, inference_time=10):
    queue = []
    dispatch_times = []
    completion_times = []
    
    for req_id, arrival_time in enumerate(request_arrivals):
        queue.append(arrival_time)
        
        if len(queue) >= batch_size:
            dispatch_time = max(queue)
            batch_completion = dispatch_time + inference_time
            
            for _ in range(batch_size):
                dispatch_times.append(dispatch_time)
                completion_times.append(batch_completion)
            
            queue = []
    
    if queue:
        dispatch_time = max(queue)
        batch_completion = dispatch_time + inference_time
        for _ in range(len(queue)):
            dispatch_times.append(dispatch_time)
            completion_times.append(batch_completion)
    
    return np.array(dispatch_times), np.array(completion_times)

np.random.seed(42)
inter_arrivals = np.random.exponential(scale=0.5, size=256)
arrivals = np.cumsum(inter_arrivals)

dispatch_static, complete_static = simulate_static_batch(arrivals, batch_size=32)
latencies_static = complete_static - dispatch_static[:len(complete_static)]

print(f'Static batching (batch_size=32):')
print(f'  Mean latency: {latencies_static.mean():.1f} ms')
print(f'  P99 latency: {np.percentile(latencies_static, 99):.1f} ms')
print(f'  Throughput: {1000 / latencies_static.mean() * 32:.1f} req/sec')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(latencies_static, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(latencies_static.mean(), color='red', linestyle='--', label=f'Mean: {latencies_static.mean():.1f}ms')
axes[0].axvline(np.percentile(latencies_static, 99), color='orange', linestyle='--', label=f'P99: {np.percentile(latencies_static, 99):.1f}ms')
axes[0].set_xlabel('Latency (ms)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Static Batching Latency')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

axes[1].scatter(arrivals, np.arange(len(arrivals)), alpha=0.5, s=20, label='Arrivals')
axes[1].axvline(dispatch_static[0], color='red', linestyle='--', linewidth=2, label='First dispatch')
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Request ID')
axes[1].set_title('Request Arrivals Timeline')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Level 2: Dynamic Batching

In [ ]:
class DynamicBatchScheduler:
    def __init__(self, min_batch=4, max_wait_ms=50, inference_time=10):
        self.min_batch = min_batch
        self.max_wait_ms = max_wait_ms
        self.inference_time = inference_time
        self.queue = deque()
        self.queue_open_time = None
        self.dispatches = []
    
    def add_request(self, request_id, arrival_time):
        if not self.queue:
            self.queue_open_time = arrival_time
        
        self.queue.append((request_id, arrival_time))
        
        should_dispatch = False
        reason = None
        
        if len(self.queue) >= self.min_batch:
            should_dispatch = True
            reason = 'min_batch'
        
        time_in_queue = arrival_time - self.queue_open_time
        if time_in_queue >= self.max_wait_ms and len(self.queue) > 0:
            should_dispatch = True
            reason = 'max_wait'
        
        if should_dispatch:
            batch = list(self.queue)
            dispatch_time = arrival_time
            completion_time = dispatch_time + self.inference_time
            
            self.dispatches.append({
                'batch_size': len(batch),
                'dispatch_time': dispatch_time,
                'completion_time': completion_time,
                'requests': batch,
                'reason': reason
            })
            
            self.queue.clear()
            self.queue_open_time = None
    
    def compute_metrics(self):
        latencies = []
        for dispatch in self.dispatches:
            for req_id, arrival in dispatch['requests']:
                latency = dispatch['completion_time'] - arrival
                latencies.append(latency)
        return np.array(latencies)

scheduler_dyn = DynamicBatchScheduler(min_batch=8, max_wait_ms=50)
for req_id, arrival in enumerate(arrivals):
    scheduler_dyn.add_request(req_id, arrival)

latencies_dyn = scheduler_dyn.compute_metrics()

print(f'Dynamic batching (min_batch=8, max_wait=50ms):')
print(f'  Mean latency: {latencies_dyn.mean():.1f} ms')
print(f'  P99 latency: {np.percentile(latencies_dyn, 99):.1f} ms')
print(f'  Mean batch size: {np.mean([d["batch_size"] for d in scheduler_dyn.dispatches]):.1f}')
print(f'  Throughput: {1000 / latencies_dyn.mean() * np.mean([d["batch_size"] for d in scheduler_dyn.dispatches]):.1f} req/sec')

In [ ]:
configs = [
    {'min_batch': 4, 'max_wait': 30, 'label': 'min=4, wait=30ms'},
    {'min_batch': 8, 'max_wait': 50, 'label': 'min=8, wait=50ms'},
    {'min_batch': 16, 'max_wait': 100, 'label': 'min=16, wait=100ms'},
]

results = []
for config in configs:
    scheduler = DynamicBatchScheduler(min_batch=config['min_batch'], max_wait_ms=config['max_wait'])
    for req_id, arrival in enumerate(arrivals):
        scheduler.add_request(req_id, arrival)
    
    lats = scheduler.compute_metrics()
    mean_batch = np.mean([d['batch_size'] for d in scheduler.dispatches])
    
    results.append({
        'config': config['label'],
        'mean_latency': lats.mean(),
        'p99_latency': np.percentile(lats, 99),
        'mean_batch': mean_batch,
        'throughput': 1000 / lats.mean() * mean_batch
    })

print('\nDynamic Batching Configs Comparison:')
print('-' * 80)
for r in results:
    print(f"{r['config']:20s} | Lat: {r['mean_latency']:6.1f}ms P99: {r['p99_latency']:6.1f}ms | Batch: {r['mean_batch']:5.1f} | Thr: {r['throughput']:7.0f} req/s")
print('-' * 80)

## Real-World Examples

In [ ]:
# Example 1: Load-dependent behavior
def generate_poisson_arrivals(arrival_rate_req_per_sec, duration_ms):
    inter_arr_ms = np.random.exponential(1000 / arrival_rate_req_per_sec, size=500)
    arrivals = np.cumsum(inter_arr_ms)
    return arrivals[arrivals < duration_ms]

low_load = generate_poisson_arrivals(10, 500)
high_load = generate_poisson_arrivals(100, 500) + 500
low_load_2 = generate_poisson_arrivals(10, 500) + 1000

all_arrivals = np.concatenate([low_load, high_load, low_load_2])

scheduler_adaptive = DynamicBatchScheduler(min_batch=4, max_wait_ms=50)
for req_id, arrival in enumerate(all_arrivals):
    scheduler_adaptive.add_request(req_id, arrival)

print('Load-dependent batching behavior:')
print(f'Phase 1 (low load): Mean batch = {np.mean([d["batch_size"] for d in scheduler_adaptive.dispatches if d["dispatch_time"] < 500]):.1f}')
print(f'Phase 2 (high load): Mean batch = {np.mean([d["batch_size"] for d in scheduler_adaptive.dispatches if 500 <= d["dispatch_time"] < 1000]):.1f}')
print(f'Phase 3 (low load): Mean batch = {np.mean([d["batch_size"] for d in scheduler_adaptive.dispatches if d["dispatch_time"] >= 1000]):.1f}')
print('\nObservation: Dynamic batching adapts batch size to load automatically')

## Key Takeaways

In [ ]:
print('='*70)
print('DYNAMIC BATCHING BEST PRACTICES')
print('='*70)
print('\n1. Parameters:')
print('   min_batch: 4-16 (smaller = lower latency)')
print('   max_wait_ms: 30-100 (lower = lower latency)')
print('\n2. Defaults: min_batch=8, max_wait_ms=50')
print('\n3. Common mistake: Set min_batch=256 without max_wait_ms')
print('   Result: Latency spikes during low-traffic periods')
print('   Fix: Always set max_wait_ms to enforce SLA')
print('\n4. Expected improvements:')
print('   Static (no waiting) → Dynamic: 30-50% latency reduction')
print('   P99 latency: especially improved (no head-of-line blocking)')
print('='*70)

## Exercises

In [ ]:
print('Exercise 1: Tune parameters for specific SLA')
print('-' * 70)
print('Given: P99 latency SLA=100ms, high load (50 req/s), inference=10ms')
print('Find: Optimal min_batch and max_wait_ms')
print('Hint: Try min_batch=16, max_wait=50 for high load')
print('')
print('Exercise 2: Multi-Tier Batching')
print('-' * 70)
print('Route GPU requests separately from CPU fallback requests')
print('Design scheduler to batch each tier independently')
print('')
print('Success! Generated notebook 47 (dynamic-batching)')